# Entity-Sentiment Inference Launcher — full SP500 (503 tickers)

Runs the production v2.0 NER + sentiment model over Tier-1 news for the entire
S&P 500, then auto-enriches each output with ticker resolution + per-ticker
sentiment aggregation.

**Scale:** ~174k T1 articles across 503 tickers, ~13h total on an A100. The run is
designed for MULTIPLE Colab sessions: each ticker is skipped if already complete,
and a partially-done ticker resumes per-article. If Colab disconnects, just re-run
cell 5 — it fast-forwards past finished tickers and continues.

Two output files per ticker in `outputs/inference/`:
- `<TICKER>.t1_sentiment.jsonl` — raw per-surface-form entities
- `<TICKER>.t1_sentiment_enriched.jsonl` — adds `ticker` + `ticker_sentiments[]`

To run a small subset instead, set `TICKER_OVERRIDE` in cell 2.

In [ ]:
# 1. Mount Drive & check GPU
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Set project path, universe, checkpoint
import os, csv

PROJECT_PATH = "/content/drive/MyDrive/entity_sentiment_model_pipeline"

# === Run params ===
TIER         = 1            # final_source_tier filter (1 = wire-grade)
MAX_ARTICLES = None         # int for a sanity subset per ticker; None = full
BATCH_SIZE   = 16           # encoder mini-batch — 90GB GPU handles 32 (~10GB peak)
LOCAL_OUTPUT_DIR = "/content"
BULK_DIR     = "eodhd_bulk_20260518"
ENRICH       = True         # auto-run enrich_with_tickers.py after each ticker
SKIP_COMPLETED = True       # skip tickers whose enriched output already covers all T1 articles

# Universe: all 503 SP500 components (eodhd_symbol column, e.g. AAPL.US, BRK-B.US).
# To run a subset, set TICKER_OVERRIDE = ["AAPL.US", "MSFT.US"]; else leave None.
TICKER_OVERRIDE = None

UNIVERSE_CSV = f"{PROJECT_PATH}/data/raw/sp500_components_20260517.csv"
with open(UNIVERSE_CSV) as f:
    ALL_TICKERS = [row["eodhd_symbol"].strip() for row in csv.DictReader(f) if row.get("eodhd_symbol", "").strip()]
TICKERS = TICKER_OVERRIDE if TICKER_OVERRIDE else ALL_TICKERS

CHECKPOINT_PATH = f"{PROJECT_PATH}/trained_model/v2.0_20260517/model.pt"

assert os.path.exists(PROJECT_PATH), f"Not found: {PROJECT_PATH}"
assert os.path.exists(f"{PROJECT_PATH}/scripts/inference/infer_entity_sentiment.py"), "inference script missing!"
assert os.path.exists(f"{PROJECT_PATH}/scripts/postprocessing/enrich_with_tickers.py"), "enrichment script missing!"
assert os.path.exists(CHECKPOINT_PATH), f"Checkpoint missing: {CHECKPOINT_PATH}"

import torch
_ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
_corr = _ckpt.get("val_metrics", {}).get("sentiment_corr", 0)
_f1   = _ckpt.get("val_metrics", {}).get("ner_f1", 0)
print(f"Project    : {PROJECT_PATH}")
print(f"Checkpoint : epoch {_ckpt.get('epoch',-1)+1}  ner_f1={_f1:.4f}  sentiment_corr={_corr:.4f}")
print(f"Universe   : {len(TICKERS)} tickers  (override={'yes' if TICKER_OVERRIDE else 'no'})")
print(f"Params     : tier=T{TIER} batch={BATCH_SIZE} skip_completed={SKIP_COMPLETED} enrich={ENRICH}")
assert _corr > 0.45, f"Checkpoint looks stale (corr={_corr:.4f})"
del _ckpt

In [ ]:
# 3. Precompute per-ticker T1 counts (drives skip-if-complete + ETA)
import gzip, json, os, time

TICKER_T1_COUNT = {}   # ticker -> number of T1 articles in its feed
missing = []
t0 = time.time()
for i, tk in enumerate(TICKERS):
    p = f"{PROJECT_PATH}/data/raw/{BULK_DIR}/news_retiered_v4/{tk}.jsonl.gz"
    if not os.path.exists(p):
        missing.append(tk); continue
    n_t1 = 0
    with gzip.open(p, "rt", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                rec = json.loads(line)
                if rec.get("final_source_tier") == TIER:
                    title = (rec.get("title") or "").strip()
                    content = (rec.get("content") or "").strip()
                    text = (title + ". " + content).strip().strip(".").strip()
                    if text:
                        n_t1 += 1
            except json.JSONDecodeError:
                pass
    TICKER_T1_COUNT[tk] = n_t1
    if (i+1) % 50 == 0:
        print(f"  scanned {i+1}/{len(TICKERS)} tickers...")

total_t1 = sum(TICKER_T1_COUNT.values())
RATE = 3.5  # articles/sec observed on A100 at batch 32
print(f"\nTotal T1 articles: {total_t1:,} across {len(TICKER_T1_COUNT)} tickers")
print(f"Estimated full-run GPU time: ~{total_t1/RATE/3600:.1f} h (at {RATE} art/s)")
if missing:
    print(f"Missing news files ({len(missing)}): {missing[:10]}")
print(f"(scan took {time.time()-t0:.0f}s)")

In [ ]:
# 4. Install deps
!pip install -q transformers torch torchvision torchaudio
!pip install -q pytorch-crf

In [ ]:
# 5. Run inference + enrichment for every ticker (multi-session safe)
#    - Skips tickers whose enriched output already covers all T1 articles.
#    - A partially-done ticker resumes per-article (handled by the inference script).
#    - Re-run this cell after any disconnect; it fast-forwards past finished work.
import subprocess, sys, os, time, shutil
from pathlib import Path

def enriched_line_count(path):
    if not os.path.exists(path): return 0
    n = 0
    with open(path) as f:
        for line in f:
            if line.strip(): n += 1
    return n

ATTEMPTED, COMPLETED, FAILED = set(), [], []
run_t0 = time.time()
done_articles = 0
total_articles = sum(TICKER_T1_COUNT.get(tk, 0) for tk in TICKERS)

for i, tk in enumerate(TICKERS, 1):
    in_path  = f"{PROJECT_PATH}/data/raw/{BULK_DIR}/news_retiered_v4/{tk}.jsonl.gz"
    out_path = f"{PROJECT_PATH}/outputs/inference/{tk}.t1_sentiment.jsonl"
    enr_path = out_path.replace(".jsonl", "_enriched.jsonl")
    expected = TICKER_T1_COUNT.get(tk, 0)
    if not os.path.exists(in_path):
        continue

    # Fast skip: enriched output already has all T1 rows (no model load).
    if SKIP_COMPLETED and enriched_line_count(enr_path) >= expected and expected > 0:
        done_articles += expected
        continue

    ATTEMPTED.add(tk)
    elapsed = time.time() - run_t0
    eta = (total_articles - done_articles) / 3.5 / 3600
    print(f"\n[{i}/{len(TICKERS)}] {tk}  (T1={expected:,})  elapsed={elapsed/60:.0f}m  rough ETA~{eta:.1f}h")

    # Seed local staging from Drive so per-article resume survives a disconnect.
    local_out = Path(LOCAL_OUTPUT_DIR) / f"{tk}.t1_sentiment.jsonl"
    if os.path.exists(out_path) and not local_out.exists():
        shutil.copy2(out_path, str(local_out))

    cmd = ["python", "scripts/inference/infer_entity_sentiment.py",
           "--input", in_path, "--output", out_path, "--checkpoint", CHECKPOINT_PATH,
           "--tier", str(TIER), "--batch-size", str(BATCH_SIZE), "--device", "auto",
           "--local-output-dir", LOCAL_OUTPUT_DIR]
    if MAX_ARTICLES is not None:
        cmd += ["--max-articles", str(MAX_ARTICLES)]
    r = subprocess.run(cmd, cwd=PROJECT_PATH, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout.decode()[-1500:])  # tail of the log
    if r.returncode != 0:
        FAILED.append(tk); print(f"  ERROR rc={r.returncode} for {tk}; continuing"); continue

    if ENRICH:
        # Prefer local raw output (guaranteed complete) over Drive out_path (may be stale if sync failed)
        enrich_input = str(local_out) if local_out.exists() else out_path
        er = subprocess.run(["python", "scripts/postprocessing/enrich_with_tickers.py",
                             "--input", enrich_input, "--output", enr_path],
                            cwd=PROJECT_PATH, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if er.returncode != 0:
            FAILED.append(tk); print(f"  WARN enrichment failed for {tk}")
    COMPLETED.append(tk)
    done_articles += min(expected, MAX_ARTICLES) if MAX_ARTICLES else expected

print(f"\n=== Pass done. attempted={len(ATTEMPTED)} completed={len(COMPLETED)} failed={len(FAILED)} ===")
if FAILED:
    print(f"Failed tickers (re-run cell 5 to retry): {FAILED}")
print("Re-run this cell if Colab disconnected mid-run; finished tickers are skipped instantly.")

In [ ]:
# 6. Per-ticker summary across all completed runs
import json, os, statistics
rows = []
for tk in TICKERS:
    path = f"{PROJECT_PATH}/outputs/inference/{tk}.t1_sentiment_enriched.jsonl"
    local_path = f"{LOCAL_OUTPUT_DIR}/{tk}.t1_sentiment_enriched.jsonl"
    read_path = local_path if os.path.exists(local_path) else path
    if not os.path.exists(read_path):
        rows.append((tk, 0, 0, 0, None))
        continue
    n_art, n_tk_rows, n_sent_rows, sents = 0, 0, 0, []
    with open(read_path) as f:
        for line in f:
            if not line.strip(): continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            n_art += 1
            for ts in r.get('ticker_sentiments', []):
                n_tk_rows += 1
                if ts.get('sentiment') is not None:
                    sents.append(ts['sentiment'])
                    n_sent_rows += 1
    mean = statistics.mean(sents) if sents else None
    rows.append((tk, n_art, n_tk_rows, n_sent_rows, mean))

print(f"{'TICKER':<12} {'#ART':>8} {'#TK_ROWS':>10} {'#SCORED':>10} {'MEAN_SENT':>10}")
for tk, n_art, n_tk, n_sc, m in rows:
    ms = f"{m:+.4f}" if m is not None else '   -   '
    print(f"  {tk:<10} {n_art:>8,d} {n_tk:>10,d} {n_sc:>10,d} {ms:>10}")

In [ ]:
# 7. For each primary_ticker's news feed: self-sentiment (ticker rows where ticker == primary's root)
import json, os, statistics
print(f"{'PRIMARY':<12} {'SELF_TICKER':<12} {'#ART_FOUND':>10} {'SELF_SENT':>10} {'STD':>8}")
for tk in TICKERS:
    path = f"{PROJECT_PATH}/outputs/inference/{tk}.t1_sentiment_enriched.jsonl"
    local_path = f"{LOCAL_OUTPUT_DIR}/{tk}.t1_sentiment_enriched.jsonl"
    rp = local_path if os.path.exists(local_path) else path
    if not os.path.exists(rp): continue
    root_ticker = tk.split('.')[0]
    self_sents = []
    with open(rp) as f:
        for line in f:
            if not line.strip(): continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            for ts in r.get('ticker_sentiments', []):
                if ts['ticker'] == root_ticker and ts.get('sentiment') is not None:
                    self_sents.append(ts['sentiment'])
                    break
    if self_sents:
        print(f"  {tk:<10} {root_ticker:<12} {len(self_sents):>10,d} {statistics.mean(self_sents):>+10.4f} {statistics.stdev(self_sents) if len(self_sents)>1 else 0:>8.4f}")
    else:
        print(f"  {tk:<10} {root_ticker:<12} {0:>10,d} {'-':>10} {'-':>8}")

In [ ]:
# 8. Cross-ticker co-mentions: which OTHER tickers show up most in each primary's news?
import json, os
from collections import Counter

for tk in TICKERS:
    path = f"{PROJECT_PATH}/outputs/inference/{tk}.t1_sentiment_enriched.jsonl"
    local_path = f"{LOCAL_OUTPUT_DIR}/{tk}.t1_sentiment_enriched.jsonl"
    rp = local_path if os.path.exists(local_path) else path
    if not os.path.exists(rp): continue
    root_ticker = tk.split('.')[0]
    cnt = Counter()
    with open(rp) as f:
        for line in f:
            if not line.strip(): continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            for ts in r.get('ticker_sentiments', []):
                if ts['ticker'] != root_ticker:
                    cnt[ts['ticker']] += 1
    top = cnt.most_common(8)
    if top:
        print(f"\n{tk} co-mentioned tickers (top 8):")
        for t, n in top:
            print(f"    {t:<10} {n:>5,d} articles")

In [ ]:
# 9. Terminate runtime to stop billing (run when the WHOLE universe is done)
from google.colab import drive, runtime
import os

try:
    drive.flush_and_unmount(); print("Drive flushed.")
except Exception as e:
    print(f"flush_and_unmount: {e}")

# How many tickers are fully done on Drive?
done = 0; pending = []
for tk in TICKERS:
    enr = f"{PROJECT_PATH}/outputs/inference/{tk}.t1_sentiment_enriched.jsonl"
    exp = TICKER_T1_COUNT.get(tk, 0)
    if exp == 0:
        continue
    n = 0
    if os.path.exists(enr):
        with open(enr) as f:
            n = sum(1 for line in f if line.strip())
    if n >= exp:
        done += 1
    else:
        pending.append((tk, n, exp))

print(f"Complete on Drive: {done} tickers")
if pending:
    print(f"Still pending: {len(pending)} (e.g. {pending[:5]})")
    print("Do NOT terminate yet — re-run cell 5 to finish them.")
else:
    print("All tickers complete. Terminating runtime.")
    runtime.unassign()